In [17]:
import os
import re
import pandas as pd
import chromadb

In [18]:
client = chromadb.HttpClient(host="localhost", port=8000)
collection_name = "heartopia_knowledge"

In [19]:
try:
    client.delete_collection(name=collection_name)
except Exception:
    pass
collection = client.create_collection(name=collection_name)

In [20]:
csv_files = ["bird.csv", "cook.csv", "crops.csv", "fish.csv", "forage.csv", "insect.csv"]

In [ ]:
for file_name in csv_files:
    if not os.path.exists(file_name):
        continue

    df = pd.read_csv(file_name, encoding="latin1")
    category_name = file_name.replace('.csv', '')
    
    documents = []
    metadatas = []
    ids = []
    
    for index, row in df.iterrows():
        valid_items = []
        for col, val in row.items():
            if pd.notna(val) and str(val).strip() not in ['nan', '-']:
                col_str = str(col).strip()
                col_lower = col_str.lower()
                val_str = str(val).strip()

                if 'Massimo' in col_lower:
                    valid_items.append(f"Available at Massimo's Store: {val_str}")
                elif 'Doris' in col_lower or 'Doris' in col_lower:
                    valid_items.append(f"Available at Doris's Store: {val_str}")
                elif col_lower == 'Crops':
                    valid_items.append(f"Crops sold by Blanc: {val_str}")
                elif 'star' in col_lower or 'rating' in col_lower:
                    match = re.search(r'([1-5])', col_lower)
                    if match:
                        star_num = match.group(1)
                        if 'profit' in col_lower:
                            valid_items.append(f"{star_num}-Star Profit Margin: {val_str}")
                        else:
                            valid_items.append(f"{star_num}-Star Price: {val_str}")
                    else:
                        valid_items.append(f"{col_str.title()}: {val_str}")
                else:
                    valid_items.append(f"{col_str.capitalize()}: {val_str}")
        
        if len(valid_items) < 2:
            continue
            
        row_details = ", ".join(valid_items)
        chunk_text = f"The {category_name} item details are: {row_details}."
        
        documents.append(chunk_text)
        metadatas.append({"category": category_name})
        ids.append(f"{category_name}_{index}")

    if documents:
        collection.add(documents=documents, metadatas=metadatas, ids=ids)

In [ ]:
# Quick Validation Check
print("--- TESTING CROP STORE SEPARATION ---")

test_queries = [
    "What crop ingredients does Massimo sell?",
    "What crop ingredients does Doris sell?"
]

for query in test_queries:
    print(f"\n❓ Query: '{query}'")
    results = collection.query(
        query_texts=[query],
        n_results=2,
        where={"category": "crops"} # Forces it to only look at crops.csv
    )
    
    for doc in results['documents'][0]:
        print(f"  👉 {doc}")

--- TESTING CROP STORE SEPARATION ---

❓ Query: 'What crop ingredients does Massimo sell?'
  👉 The crops item details are: Crops: Cocoa, Level: 12, Growth time: 5h, Seed cost: 110, 1-Star Price: 330, 2-Star Price: 442, 3-Star Price: 551, 4-Star Price: 660, 5-Star Price: 990, 1-Star Profit Margin: 67%, 2-Star Profit Margin: 75%, 3-Star Profit Margin: 80%, 4-Star Profit Margin: 83%, 5-Star Profit Margin: 89%, Available at Massimo's Store: PasturizedEgg, Discounted price: 60.0, Base price: 100.0.
  👉 The crops item details are: Crops: TeaTree, Level: 11, Growth time: 45m, Seed cost: 25, 1-Star Price: 75, 2-Star Price: 100, 3-Star Price: 125, 4-Star Price: 150, 5-Star Price: 225, 1-Star Profit Margin: 67%, 2-Star Profit Margin: 75%, 3-Star Profit Margin: 80%, 4-Star Profit Margin: 83%, 5-Star Profit Margin: 89%, Available at Massimo's Store: CookingOil, Discounted price: 60.0, Base price: 100.0.

❓ Query: 'What crop ingredients does Dorris sell?'
  👉 The crops item details are: Crops: Lett